# CountZES — Spike Test & Playground (Google Colab)

Notebook này dùng để **thử nghiệm tự do (spike test)** trên bất kỳ ảnh nào (ảnh từ URL, ảnh upload từ máy, hoặc ảnh có sẵn) với prompt class bất kỳ (`"cars"`, `"apples"`, `"cells"`, `"people"`, ...), đồng thời **vẽ ra toàn bộ các bước trung gian**:
- CLIP similarity map
- GroundingDINO detections
- 3 Exemplar được chọn: **DAE** (Stage 1), **DGE** (Stage 2), **FCE** (Stage 3)
- Density Map dự đoán & so sánh A/B với cờ P2 (`--fix_sses_inversion`, `--single_instance_mode`)

In [ ]:
# [CELL-1] Thiết lập môi trường, clone/pull repo & cài thư viện
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

REPO_URL = "https://github.com/duylw/cv-advanced.git"
REPO_DIR = "/content/cv-advanced"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull origin main

!pip install -q opencv-python tqdm matplotlib timm einops supervision sentencepiece accelerate scipy pandas scikit-image addict yapf pycocotools gdown
!pip install -q "transformers>=4.31.0"
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!nvidia-smi

In [ ]:
# [CELL-2] Đảm bảo có sẵn weights (SAM ViT-H, DSALVANet, GroundingDINO)
!mkdir -p data DSALVANet/checkpoints GroundingDINO/weights CounTR/output_allnew_dir

if not os.path.exists("data/sam_vit_h_4b8939.pth"):
    !wget -q --show-progress -O data/sam_vit_h_4b8939.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
if not os.path.exists("DSALVANet/checkpoints/checkpoint_200.pth"):
    !gdown 1julzH9MJSK1xTGchb1r0CXdZ2wzF5-Kp -O DSALVANet/checkpoints/checkpoint_200.pth
if not os.path.exists("GroundingDINO/weights/groundingdino_swint_ogc.pth"):
    !gdown 13rV6dzRwWCVZYWpnmiaVwRDIDC28d82g -O GroundingDINO/weights/groundingdino_swint_ogc.pth

In [ ]:
# [CELL-3] Load toàn bộ 5 model 1 lần vào GPU để test nhiều ảnh liên tiếp không cần reload
from types import SimpleNamespace
from countzes_core import load_pipeline_models

FSOC_BACKEND = "DSALVANet"  # Đổi thành "countr" nếu đã đặt checkpoint vào CounTR/output_allnew_dir/
args = SimpleNamespace(
    data="./data",
    outdir="spike_test",
    ckpt=None,
    sam_type="vit_h",
    ref_idx="00",
    visualize=True,
    fsoc=FSOC_BACKEND,
    weight="./DSALVANet/checkpoints/checkpoint_200.pth",
    countr_ckpt="./CounTR/output_allnew_dir/FSC147.pth",
    fix_sses_inversion=False,
    single_instance_mode="paper",
)
models = load_pipeline_models(args)
print("Models loaded on:", models["device"])

In [ ]:
# [CELL-4] Lấy 1 ảnh thử nghiệm bất kỳ (từ URL hoặc upload trực tiếp từ máy)
import urllib.request

TEST_IMAGE_URL = "https://ultralytics.com/images/bus.jpg"  # Thay bằng URL ảnh bất kỳ bạn muốn test
CLASS_PROMPT = "person"                                   # Thay tên object cần đếm (vd: "bus", "car", "apple", "coin")

os.makedirs("outputs/spike_inputs", exist_ok=True)
TEST_IMAGE_PATH = "outputs/spike_inputs/test_sample.jpg"
urllib.request.urlretrieve(TEST_IMAGE_URL, TEST_IMAGE_PATH)
print(f"Saved test image -> {TEST_IMAGE_PATH} | Prompt: '{CLASS_PROMPT}'")

# --- Nếu muốn upload ảnh từ máy cá nhân trên Colab Web, bỏ comment 3 dòng dưới ---
# from google.colab import files
# uploaded = files.upload()
# TEST_IMAGE_PATH = list(uploaded.keys())[0]

In [ ]:
# [CELL-5] Chạy nhanh CountZES và xem kết quả cuối
import cv2
import matplotlib.pyplot as plt
from countzes_core import run_countzes_on_image

vis_output, pred_count = run_countzes_on_image(
    image_path=TEST_IMAGE_PATH,
    filename=os.path.basename(TEST_IMAGE_PATH),
    class_name=CLASS_PROMPT,
    models=models,
    args=args,
)

print(f"==> Predicted Count for '{CLASS_PROMPT}': {pred_count}")
plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(vis_output, cv2.COLOR_BGR2RGB))
plt.title(f"CountZES ({args.fsoc}) | Prompt='{CLASS_PROMPT}' | Pred={pred_count}")
plt.axis("off")
plt.show()

In [ ]:
# [CELL-6] Soi chi tiết từng giai đoạn (CLIP map + GroundingDINO + 3 Exemplars + A/B P2 flags)
from countzes_core import compute_clip_similarity_map, normalize_sim_map

src_img = cv2.imread(TEST_IMAGE_PATH)
img_rgb = cv2.cvtColor(src_img, cv2.COLOR_BGR2RGB)

# 1. CLIP similarity map
sim_clip = normalize_sim_map(
    compute_clip_similarity_map(
        img_rgb, CLASS_PROMPT, models["clip_model"], models["clip_processor"], models["device"]
    )
)

# 2. GroundingDINO boxes
det = models["grounding_dino"].predict_with_classes(
    image=src_img, classes=[CLASS_PROMPT], box_threshold=0.20, text_threshold=0.15
)
vis_det = img_rgb.copy()
if det is not None and len(det.xyxy) > 0:
    for (x0, y0, x1, y1), conf in zip(det.xyxy, det.confidence):
        cv2.rectangle(vis_det, (int(x0), int(y0)), (int(x1), int(y1)), (0, 255, 0), 2)
        cv2.putText(vis_det, f"{conf:.2f}", (int(x0), max(15, int(y0) - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

# 3. So sánh A/B: mặc định (paper) vs bật cờ sửa P2
args_ab = SimpleNamespace(**vars(args))
args_ab.fix_sses_inversion = True
args_ab.single_instance_mode = "unit"
vis_ab, pred_ab = run_countzes_on_image(
    image_path=TEST_IMAGE_PATH,
    filename=os.path.basename(TEST_IMAGE_PATH),
    class_name=CLASS_PROMPT,
    models=models,
    args=args_ab,
)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].imshow(sim_clip, cmap="jet"); axes[0].set_title("1. CLIP Similarity Map")
axes[1].imshow(vis_det); axes[1].set_title(f"2. GroundingDINO ({len(det.xyxy) if det is not None else 0} boxes)")
axes[2].imshow(cv2.cvtColor(vis_output, cv2.COLOR_BGR2RGB)); axes[2].set_title(f"3. Default (Paper) -> {pred_count}")
axes[3].imshow(cv2.cvtColor(vis_ab, cv2.COLOR_BGR2RGB)); axes[3].set_title(f"4. A/B P2 Fix -> {pred_ab}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()